<a href="https://colab.research.google.com/github/jb-612/cursor-analytics-platform/blob/main/dox_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

file_path = '/content/dev-june/Amdocs_PR_Jun_2026.xlsx'
df = pd.read_excel(file_path)

In [ ]:
print('First 5 rows of the DataFrame:')
display(df.head())

print('\nDataFrame information:')
display(df.info())

## Basic Statistics and Data Cleansing

In [ ]:
# Display descriptive statistics for numerical columns
print('\nDescriptive Statistics for Numerical Columns:')
display(df.describe())

In [ ]:
# Check for missing values
print('\nMissing values before cleansing:')
display(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

### Data Cleansing: Handling Missing Values and Data Types

Based on the `df.info()` output and the missing values check:

1.  **`labels`**: This column is entirely null. It will be dropped.
2.  **Date Columns**: `created_at`, `merged_at`, `closed_at`, `first_commit_at`, `first_review_at` will be converted to datetime objects. Missing values in `merged_at` and `closed_at` might indicate PRs that are still open or not yet closed/merged, so we'll keep them as `NaT` (Not a Time) after conversion.
3.  **Time-related numerical columns**: `coding_lead_time_hours`, `pickup_time_hours`, `review_lead_time_hours`, `total_cycle_time_hours` have missing values. For initial analysis, we'll fill these with 0, assuming that a missing value means no time was spent in that phase (e.g., no review lead time if there was no review). However, depending on the specific analysis, a more sophisticated imputation strategy might be needed.
4.  **`merge_week` and `merge_month`**: Convert to appropriate categorical types or derive from `merged_at` if necessary. For now, we'll keep them as objects but address their nulls.

In [ ]:
# Drop the 'labels' column as it's entirely null
if 'labels' in df.columns:
    df = df.drop(columns=['labels'])
    print("Dropped 'labels' column due to all null values.")

# Convert date columns to datetime objects
date_cols = ['created_at', 'merged_at', 'closed_at', 'first_commit_at', 'first_review_at']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f"Converted '{col}' to datetime.")

# Fill NaN in time-related numerical columns with 0
time_numerical_cols = [
    'coding_lead_time_hours',
    'pickup_time_hours',
    'review_lead_time_hours',
    'total_cycle_time_hours'
]
for col in time_numerical_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)
        print(f"Filled NaN in '{col}' with 0.")

# For 'merge_week' and 'merge_month', fill NaNs with a placeholder if they exist (e.g., 'Unknown')
# Or, if merged_at is null, then these should also be null. For now, fill with 'Unknown' for object type.
if 'merge_week' in df.columns:
    df['merge_week'] = df['merge_week'].fillna('Unknown')
    print("Filled NaN in 'merge_week' with 'Unknown'.")
if 'merge_month' in df.columns:
    df['merge_month'] = df['merge_month'].fillna('Unknown')
    print("Filled NaN in 'merge_month' with 'Unknown'.")

# Display missing values after cleansing
print('\nMissing values after cleansing:')
display(df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False))

# Display updated DataFrame information to confirm data types and non-null counts
print('\nDataFrame information after cleansing:')
display(df.info())

In [ ]:
# Display the first few rows of the cleaned DataFrame
print('\nFirst 5 rows of the DataFrame after cleansing:')
display(df.head())

## Filtering for Pure Product Code Projects

To focus on 'pure product code projects' and exclude 'scripting, terraform, helm charts, configuration, and devops infra', we'll filter the `repository` column based on keywords.

In [ ]:
# Define keywords for inclusion and exclusion
include_keywords = ['java', 'c\\+\\+', 'cpp', 'c#', 'dotnet', 'product', 'backend', 'frontend'] # Corrected escaping for c++
exclude_keywords = [
    'script', 'terraform', 'helm', 'config', 'configuration',
    'infra', 'infrastructure', 'devops', 'e2e-test', 'playwright',
    'utility', 'tool', 'doc', 'test', 'build', 'pipeline'
]

# Let's define the primary inclusion based on all include_keywords
# This will be more general to capture 'product code' even if the language isn't explicit
language_include_mask = df['repository'].str.contains('|'.join(include_keywords), case=False, na=False)

# Define secondary exclusion for common non-product code types
non_product_exclude_mask = df['repository'].str.contains('|'.join(exclude_keywords), case=False, na=False)

# Filter the DataFrame: include if it matches an inclusion keyword AND does not match an exclusion keyword
filtered_df = df[language_include_mask & ~non_product_exclude_mask]

print(f"Original DataFrame shape: {df.shape}")
print(f"Filtered DataFrame shape (product code only): {filtered_df.shape}")

print('\nFirst 5 rows of the filtered DataFrame:')
display(filtered_df.head())

print('\nFiltered DataFrame information:')
display(filtered_df.info())

## Distribution of Total Cycle Time Hours

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(filtered_df['total_cycle_time_hours'], bins=50, kde=True)
plt.title('Distribution of Total Cycle Time Hours for Product Code PRs')
plt.xlabel('Total Cycle Time Hours')
plt.ylabel('Number of Pull Requests')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()